# IEMOCAP — frozen multimodal-teacher feature extraction (local smoke test)

Goal: confirm the **Qwen2.5-Omni-3B (4-bit)** teacher extraction runs **locally**
(e.g. an RTX 3060) on IEMOCAP, with **video fed as a few sub-sampled frames only**.

Design mirrors `src/mintrec/teacher_probe/extract_features.py`:
- `prompt_first` single forward pass, content order **[text, video-frames, audio]**,
  audio **last** so its tokens absorb the text+video context under causal attention.
- `use_audio_in_video=False`; audio is the per-utterance `sentences/wav` clip.
- Pool **audio_mean** over the audio-token block at layers `[24, 27, 30, 34]` (+ mean).

IEMOCAP specifics handled here:
- labels = 4-class single-label (`ang / hap(+exc) / sad / neu`),
- video is **dialog-level**, so we seek into the dialog `.avi` and sample frames
  inside each utterance's `[start, end]` window,
- transcript = the `dialog/transcriptions` text (the 'text' modality).

Run top-to-bottom. Adjust the config cell, eyeball the frame-viz cell (set
`SPEAKER_CROP`), then run the smoke test.

In [ ]:
# ── config ────────────────────────────────────────────────────────────────
import sys, os, re, gc, json
from pathlib import Path
import numpy as np

# Register FFmpeg DLLs on Windows (no-op elsewhere) — needed by librosa/cv2 backends.
if sys.platform == "win32":
    for _p in os.environ.get("PATH", "").split(";"):
        if _p and os.path.exists(os.path.join(_p, "avcodec-62.dll")):
            os.add_dll_directory(_p)
            break

# Locate src/ and import the shared path config.
_cands = [Path.cwd(), *Path.cwd().parents]
_SRC = next((p for p in _cands if p.name == "src"), None)
if _SRC is None:
    _SRC = next((p / "src" for p in _cands if (p / "src" / "common" / "config.py").exists()), None)
assert _SRC is not None, "cannot locate src/ — run this notebook from inside the repo"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))
from common.config import IEMOCAP_DATA, IEMOCAP_OUTPUTS   # noqa: E402

# ── teacher / extraction settings (kept close to MIntRec) ──
MODEL_NAME = "Qwen/Qwen2.5-Omni-3B"
DTYPE      = "4bit"              # local 3060 default; "bf16" on a big GPU
LLM_LAYERS = [24, 27, 30, 34]
MEAN_COMBO = "L24-27-30-34"
ADD_GEN_PROMPT = True

# ── video sub-sampling (the extreme-edge knob) ──
NUM_VIDEO_FRAMES = 4            # MUST be even (Qwen temporal patch = 2). 2 if VRAM-tight, 8 if plenty.
FRAME_MAX_SIDE   = 336          # resize longest side -> caps vision tokens (~27/frame)
SPEAKER_CROP     = "full"       # "full" | "left" | "right"  (decide via the viz cell)
AUDIO_SR         = 16000

# ── labels: IEMOCAP 4-class benchmark (merge exc->hap, drop the rest + 'xxx') ──
EMO_MAP  = {"ang": "ang", "hap": "hap", "exc": "hap", "sad": "sad", "neu": "neu"}
LABELS   = ["ang", "hap", "neu", "sad"]
label2id = {l: i for i, l in enumerate(LABELS)}

IEMOCAP_ROOT = IEMOCAP_DATA / "IEMOCAP_full_release"
FEAT_TAG = f"iemocap_4class__qwen2.5-omni-3b-{DTYPE}__pf_text-video{NUM_VIDEO_FRAMES}f-audio__audiomean"
OUT_DIR  = IEMOCAP_DATA / "teacher_features" / FEAT_TAG

AUDIO_START_ID_DEFAULT = 151647
AUDIO_END_ID_DEFAULT   = 151648

TASK_PROMPT_TEMPLATE = (
    "You are analyzing a short clip from a dyadic conversation to recognize the "
    "speaker's emotion among 4 classes (angry, happy, neutral, sad). "
    'The transcript of what is said is: "{text}".'
)

print("IEMOCAP_ROOT:", IEMOCAP_ROOT, "| exists:", IEMOCAP_ROOT.exists())
print("FEAT_TAG    :", FEAT_TAG)

In [ ]:
# ── IEMOCAP index: parse EmoEvaluation + transcripts ──────────────────────
import pandas as pd

# category lines look like:  [6.2901 - 8.2357]\tSes01F_impro01_F000\tneu\t[2.5, 2.5, 2.5]
LINE_RE = re.compile(r"^\[(\d+\.\d+)\s*-\s*(\d+\.\d+)\]\s+(\S+)\s+(\w+)\s+\[([-\d.,\s]+)\]")

def parse_transcript(dialog, session):
    p = IEMOCAP_ROOT / f"Session{session}" / "dialog" / "transcriptions" / f"{dialog}.txt"
    d = {}
    if p.exists():
        for line in p.read_text(encoding="utf-8", errors="ignore").splitlines():
            m = re.match(r"^(\S+)\s+\[[^\]]*\]:\s*(.*)$", line)
            if m:
                d[m.group(1)] = m.group(2).strip()
    return d

def build_index():
    rows = []
    for s in range(1, 6):
        emo_dir = IEMOCAP_ROOT / f"Session{s}" / "dialog" / "EmoEvaluation"
        if not emo_dir.exists():
            continue
        for f in sorted(emo_dir.glob("*.txt")):
            dialog = f.stem
            tx = parse_transcript(dialog, s)
            for line in f.read_text(encoding="utf-8", errors="ignore").splitlines():
                m = LINE_RE.match(line.strip())
                if not m:
                    continue
                start, end, utt, emo, vad = m.groups()
                if emo not in EMO_MAP:
                    continue
                lab = EMO_MAP[emo]
                if lab not in label2id:
                    continue
                vad = [float(x) for x in vad.replace(" ", "").split(",") if x != ""]
                v, a, dom = (vad + [None, None, None])[:3]
                rows.append({
                    "utt": utt, "session": s, "dialog": dialog,
                    "start": float(start), "end": float(end),
                    "emo_raw": emo, "label": lab, "label_id": label2id[lab],
                    "V": v, "A": a, "D": dom, "text": tx.get(utt, ""),
                })
    return pd.DataFrame(rows)

def avi_path(r):
    return IEMOCAP_ROOT / f"Session{r['session']}" / "dialog" / "avi" / "DivX" / f"{r['dialog']}.avi"

def wav_path(r):
    return IEMOCAP_ROOT / f"Session{r['session']}" / "sentences" / "wav" / r["dialog"] / f"{r['utt']}.wav"

df = build_index()
print(f"{len(df)} utterances")
if len(df):
    print("class counts:", df["label"].value_counts().to_dict())
    print("duration s  :", round((df["end"] - df["start"]).mean(), 2), "mean /",
          round((df["end"] - df["start"]).max(), 2), "max")
df.head()

In [ ]:
# ── frame sampling (cv2) + audio load ─────────────────────────────────────
import cv2, librosa
from PIL import Image

def _crop(img, mode):
    if mode == "full":
        return img
    w = img.shape[1]
    half = w // 2
    return img[:, :half] if mode == "left" else img[:, half:]

def _resize_max_side(img, max_side):
    h, w = img.shape[:2]
    sc = max_side / max(h, w)
    if sc < 1.0:
        img = cv2.resize(img, (int(round(w * sc)), int(round(h * sc))), interpolation=cv2.INTER_AREA)
    return img

def extract_frames(path, start, end, n_frames=None, crop=None, max_side=None):
    # Sample n interior frames in [start, end] from the dialog avi; returns list[PIL.Image] (RGB).
    n_frames = n_frames or NUM_VIDEO_FRAMES
    crop = crop or SPEAKER_CROP
    max_side = max_side or FRAME_MAX_SIDE
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise FileNotFoundError(f"cannot open video: {path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    if end <= start:
        end = start + 1.0 / fps
    times = np.linspace(start, end, n_frames + 2)[1:-1]   # interior, avoid boundaries
    frames = []
    for t in times:
        idx = int(round(t * fps))
        if total:
            idx = min(idx, total - 1)
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, fr = cap.read()
        if not ok:
            continue
        fr = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
        fr = _resize_max_side(_crop(fr, crop), max_side)
        frames.append(Image.fromarray(fr))
    cap.release()
    if not frames:
        raise RuntimeError(f"no frames extracted: {path} [{start}, {end}]")
    if len(frames) % 2 == 1:                # Qwen needs an even frame count
        frames = frames[:-1] if len(frames) > 1 else frames + frames
    return frames

def load_audio(path):
    wav, _ = librosa.load(str(path), sr=AUDIO_SR, mono=True)
    return wav

print("frame/audio helpers ready")

In [ ]:
# ── VISUAL CHECK: eyeball one utterance, then set SPEAKER_CROP above ───────
# If both actors are in frame, switch SPEAKER_CROP to 'left'/'right' and re-run.
import matplotlib.pyplot as plt

assert len(df), "index is empty — is IEMOCAP placed at IEMOCAP_ROOT?"
r = df.iloc[0]
print("avi :", avi_path(r), "| exists:", avi_path(r).exists())
print("wav :", wav_path(r), "| exists:", wav_path(r).exists())

frames = extract_frames(avi_path(r), r["start"], r["end"])
wav = load_audio(wav_path(r))
print(f"{len(frames)} frames, size {frames[0].size}; audio {len(wav)/AUDIO_SR:.2f}s")

fig, axes = plt.subplots(1, len(frames), figsize=(4 * len(frames), 4))
for ax, im in zip(np.atleast_1d(axes), frames):
    ax.imshow(im); ax.axis("off")
plt.suptitle(f"{r['utt']}  emo={r['label']}  [{r['start']:.1f}-{r['end']:.1f}]  | {r['text'][:70]}")
plt.tight_layout(); plt.show()

In [ ]:
# ── load the 4-bit teacher + extraction helpers (mirrors MIntRec) ─────────
import torch
from transformers import (
    BitsAndBytesConfig, Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor,
)
from qwen_omni_utils import process_mm_info

def load_teacher(dtype="4bit"):
    kw = dict(device_map="auto", attn_implementation="sdpa")
    if dtype == "4bit":
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        )
    else:
        kw["torch_dtype"] = torch.bfloat16
    proc = Qwen2_5OmniProcessor.from_pretrained(MODEL_NAME)
    mdl = Qwen2_5OmniForConditionalGeneration.from_pretrained(MODEL_NAME, **kw)
    mdl.eval()
    return mdl, proc

def get_special_id(model, name, default):
    cfg = model.config
    if hasattr(cfg, name):
        return getattr(cfg, name)
    for sub in ("thinker_config", "talker_config"):
        if hasattr(cfg, sub) and hasattr(getattr(cfg, sub), name):
            return getattr(getattr(cfg, sub), name)
    return default

def find_audio_indices(input_ids, start_id, end_id):
    ids = input_ids[0].tolist()
    s, e = ids.index(start_id), ids.index(end_id)
    idx = list(range(s + 1, e))
    if not idx:
        raise ValueError("no audio tokens between start/end markers")
    return idx, s, e

def _cpu16(t):
    return t.detach().float().cpu().to(torch.float16)

def pool_audio_mean(hidden_states, audio_idx):
    feats, per_layer = {}, []
    for L in LLM_LAYERS:
        v = hidden_states[L][0][audio_idx].mean(dim=0)
        feats[f"pf_audio_mean_l{L}"] = _cpu16(v)
        per_layer.append(v)
    feats[f"pf_audio_mean_{MEAN_COMBO}"] = _cpu16(torch.stack(per_layer, 0).mean(0))
    return feats

def build_inputs(processor, model, frames, wav, transcript):
    # content order = [text, video(our sampled frames), audio]; audio LAST.
    conversation = [{"role": "user", "content": [
        {"type": "text",  "text": TASK_PROMPT_TEMPLATE.format(text=transcript)},
        {"type": "video", "video": frames},     # list[PIL] = pre-sampled frames
        {"type": "audio", "audio": wav},
    ]}]
    text_input = processor.apply_chat_template(conversation, add_generation_prompt=ADD_GEN_PROMPT, tokenize=False)
    audios, images, videos = process_mm_info(conversation, use_audio_in_video=False)
    inputs = processor(text=text_input, audio=audios, images=images, videos=videos,
                       return_tensors="pt", padding=True, use_audio_in_video=False).to(model.device)
    return inputs

@torch.no_grad()
def extract_sample(model, processor, frames, wav, transcript, a_start, a_end):
    inputs = build_inputs(processor, model, frames, wav, transcript)
    out = model.thinker(**inputs, output_hidden_states=True, return_dict=True)
    audio_idx, s, e = find_audio_indices(inputs["input_ids"], a_start, a_end)
    feats = pool_audio_mean(out.hidden_states, audio_idx)
    meta = {"seq_len": int(inputs["input_ids"].shape[1]),
            "num_audio_tokens": len(audio_idx), "audio_range": [s + 1, e]}
    return feats, meta

print("loading teacher ...")
model, processor = load_teacher(DTYPE)
A_START = get_special_id(model, "audio_start_token_id", AUDIO_START_ID_DEFAULT)
A_END   = get_special_id(model, "audio_end_token_id", AUDIO_END_ID_DEFAULT)
print("audio_start/end:", A_START, A_END)

In [ ]:
# ── SMOKE TEST: 5 utterances — check it runs, locate audio tokens, peak VRAM
import torch
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

for i in range(min(5, len(df))):
    r = df.iloc[i]
    try:
        frames = extract_frames(avi_path(r), r["start"], r["end"])
        wav = load_audio(wav_path(r))
        feats, meta = extract_sample(model, processor, frames, wav, r["text"], A_START, A_END)
        dim = feats[f"pf_audio_mean_{MEAN_COMBO}"].shape[0]
        print(f"[{i}] {r['utt']:<22} emo={r['label']:<3} seq={meta['seq_len']:<5} "
              f"audio_toks={meta['num_audio_tokens']:<4} dim={dim}")
    except Exception as ex:
        print(f"[{i}] {r['utt']} FAILED: {type(ex).__name__}: {ex}")

if torch.cuda.is_available():
    print(f"peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

In [ ]:
# ── MINI EXTRACTION: first LIMIT samples -> smoke .pt (same layout as MIntRec)
from tqdm.auto import tqdm
LIMIT = 20

feat_bank, labels, sample_ids, metadata = {}, [], [], []
for i in tqdm(range(min(LIMIT, len(df))), desc="extract"):
    r = df.iloc[i]
    if not (avi_path(r).exists() and wav_path(r).exists()):
        continue
    try:
        frames = extract_frames(avi_path(r), r["start"], r["end"])
        wav = load_audio(wav_path(r))
        feats, meta = extract_sample(model, processor, frames, wav, r["text"], A_START, A_END)
    except Exception as ex:
        print("skip", r["utt"], ex); continue
    for k, v in feats.items():
        feat_bank.setdefault(k, []).append(v)
    labels.append(int(r["label_id"])); sample_ids.append(r["utt"])
    meta.update({"utt": r["utt"], "label": r["label"], "label_id": int(r["label_id"]),
                 "V": r["V"], "A": r["A"], "D": r["D"]})
    metadata.append(meta)
    if (i + 1) % 5 == 0 and torch.cuda.is_available():
        torch.cuda.empty_cache(); gc.collect()

result = {
    "labels": torch.tensor(labels, dtype=torch.long),
    "sample_ids": sample_ids, "metadata": metadata,
    "features": {k: torch.stack(v, 0) for k, v in feat_bank.items()},
    "feature_dim": (next(iter(feat_bank.values()))[0].shape[0] if feat_bank else 0),
}
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUT_DIR / "smoke_features.pt"
torch.save(result, out_path)
print(f"saved {len(labels)} samples x {len(feat_bank)} features (dim={result['feature_dim']}) -> {out_path}")

## Next steps

If the smoke test stays within VRAM and `num_audio_tokens > 0`, the local 4-bit path
works. Then:

1. **Decide `NUM_VIDEO_FRAMES` / `SPEAKER_CROP`** from the viz + VRAM numbers.
2. **Port to `extract_features.py`** — copy the sharded, resume-safe loop from
   `src/mintrec/teacher_probe/extract_features.py` (shard flush / `_load_shards` /
   `_count_done`), swapping in `build_index` + `extract_frames` from here. Add a
   train/dev/test split (IEMOCAP convention: Session 1–4 train, Session 5 test, or
   leave-one-session-out).
3. **Probe** the pooled features with `common.probe.Probe` as the go/no-go signal,
   then move to the (audio / audio-visual) student + KD, reusing `common.losses` /
   `common.training`.